01 Exploração de Dados (EDA) — Camada Raw

Primeira etapa da análise: entender o dado exatamente como ele chegou do Portal da Transparência, antes de qualquer limpeza. Aqui identificaremos problemas (nulos, texto onde deveria ser número, duplicidade, inconsistência de formato) que justificam as decisões tomadas depois, em `02_limpeza.ipynb`.

Pré-requisito: banco criado com `sql/0_criar_banco.sql` e `.env` configurado.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Permite importar o pacote `src` a partir da pasta notebooks/
sys.path.append(str(Path("..").resolve()))

from src import banco
from src.extracao import executar_extracao

executar_extracao()  # baixa o zip (se necessario) e carrega as 4 tabelas Raw


In [ ]:
conexao = banco.conectar()

def consultar(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conexao)

print("Conectado ao banco com sucesso.")


In [ ]:
tabelas_raw = ["raw_viagem", "raw_pagamento", "raw_passagem", "raw_trecho"]
volumes = {t: consultar(f"SELECT COUNT(*) AS total FROM {t}")["total"][0] for t in tabelas_raw}
pd.Series(volumes, name="linhas").to_frame()


Qualidade dos dados em `raw_viagem`

Valores ausentes/nulos nas colunas-chave
Como a Raw guarda tudo em `VARCHAR`, "nulo" aqui também inclui string vazia (`''`) — é assim que o Portal da Transparência representa a ausência de informação.

In [ ]:
sql_nulos = """
    SELECT
        SUM(id_viagem = '' OR id_viagem IS NULL)                     AS sem_id_viagem,
        SUM(nome_orgao_superior = '' OR nome_orgao_superior IS NULL) AS sem_orgao_superior,
        SUM(data_inicio = '' OR data_inicio IS NULL)                 AS sem_data_inicio,
        SUM(data_fim = '' OR data_fim IS NULL)                       AS sem_data_fim,
        SUM(valor_diarias = '' OR valor_diarias IS NULL)             AS sem_valor_diarias
    FROM raw_viagem;
"""
consultar(sql_nulos)


Duplicidade de `id_viagem` (chave primária pretendida na Silver)

In [ ]:
sql_duplicados = """
    SELECT id_viagem, COUNT(*) AS ocorrencias
    FROM raw_viagem
    WHERE id_viagem <> ''
    GROUP BY id_viagem
    HAVING COUNT(*) > 1
    ORDER BY ocorrencias DESC
    LIMIT 10;
"""
consultar(sql_duplicados)


Valores distintos em colunas categóricas (para conferir inconsistências de digitação)

In [ ]:
consultar("SELECT situacao, COUNT(*) AS qtd FROM raw_viagem GROUP BY situacao ORDER BY qtd DESC;")


In [ ]:
consultar("SELECT viagem_urgente, COUNT(*) AS qtd FROM raw_viagem GROUP BY viagem_urgente ORDER BY qtd DESC;")


Amostra de valores monetários e de datas (texto, não número)

In [ ]:
consultar("""
    SELECT valor_diarias, valor_passagens, data_inicio, data_fim
    FROM raw_viagem
    WHERE valor_diarias <> ''
    LIMIT 5;
""")


Qualidade dos dados nas demais tabelas Raw

`raw_pagamento` — tipos de pagamento cadastrados

In [ ]:
consultar("SELECT tipo_pagamento, COUNT(*) AS qtd FROM raw_pagamento GROUP BY tipo_pagamento ORDER BY qtd DESC;")


`raw_trecho` — meios de transporte cadastrados

In [ ]:
consultar("SELECT meio_transporte, COUNT(*) AS qtd FROM raw_trecho GROUP BY meio_transporte ORDER BY qtd DESC;")


Conclusões da exploração

- Existem linhas com `id_viagem` ou `nome_orgao_superior` vazios — precisam  ser descartadas na Silver, pois violam a PK e a constraint `NOT NULL`.
- Há `id_viagem` duplicado no Raw — a Silver deve manter apenas uma ocorrência  por viagem.
- Datas vêm no formato texto `DD/MM/AAAA` e valores monetários usam vírgula como separador decimal (`1.272,97`) — ambos precisam de conversão explícita
  de tipo.
- Colunas categóricas (`situacao`, `viagem_urgente`, `tipo_pagamento`, `meio_transporte`) têm um conjunto pequeno e consistente de valores, então não é necessário um dicionário de "de-para" nessas colunas.

Essas conclusões orientam diretamente as regras aplicadas em
[`02_limpeza.ipynb`](./02_limpeza.ipynb).

In [ ]:
conexao.close()
print("Conexao encerrada.")
